# Text-to-3D API server (for the local web UI): direct by default, image-based on toggle

`hunyuan3d_baseline.ipynb` generates one mesh and you download it by hand.
This notebook instead keeps models **loaded and running** as an HTTP server,
exposed through a public tunnel, so `ml/cad/frontend/index.html` (the local
prompt -> 3D viewer) can point at it directly.

Two generation modes, both served from the same endpoint via a `mode` field:

- **`"direct"` (default)** — Shap-E: genuinely text-conditioned, no image
  step, fast (seconds).
- **`"image"` (opt-in / toggle)** — the original pipeline: text -> image
  (`sd-turbo`) -> Hunyuan3D-2.1 shape. Slower (a few minutes), more detail.

**Read this before running:**
- This reuses every fix from `hunyuan3d_baseline.ipynb` for the image-based
  path (the stale-clone cleanup, the cwd/import-cache fix for `hy3dshape`) --
  if you haven't gotten that notebook to produce one mesh successfully yet,
  do that first for the image-based path specifically. The direct (Shap-E)
  path doesn't depend on any of that and is simpler to get working on its
  own.
- The public URL this creates is **unauthenticated** -- anyone who has the
  link can trigger generations on your Kaggle GPU while this notebook is
  running. Don't post it publicly; treat it like a temporary shared secret.
  It stops working the moment this notebook's session ends (closed tab,
  idle timeout, or you hit Kaggle's session/quota limit).
- All three models (Shap-E, the text-to-image model, and Hunyuan3D's shape
  pipeline) stay loaded in VRAM simultaneously so either mode responds
  without a reload -- this should fit a 16GB T4/P100 with some headroom, but
  see step 3's docstring if you hit a CUDA out-of-memory error under load.

## 0. Confirm GPU is attached

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU visible -- set Accelerator to GPU T4 x2 or P100 in the notebook settings panel, then Factory Reset the session."
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")

## 1. Install Hunyuan3D-2.1 (shape-only, same as the baseline notebook)

In [ ]:
import os
os.chdir("/kaggle/working")

if os.path.exists("/kaggle/working/Hunyuan3D-2.1"):
    !rm -rf /kaggle/working/Hunyuan3D-2.1

!git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git /kaggle/working/Hunyuan3D-2.1
%cd /kaggle/working/Hunyuan3D-2.1
!pip install -q -r hy3dshape/requirements.txt
!pip install -q diffusers accelerate fastapi uvicorn

assert os.path.exists("hy3dshape/hy3dshape/rembg.py"), \
    "Clone looks incomplete -- rerun this cell (check the git clone output above for the actual error)."
print("clone OK")

## 1b. (Optional) Hugging Face login

Only needed if step 2 fails with a "gated repo" / 401 error. See
`hunyuan3d_baseline.ipynb` for the full explanation.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(hf_token)
    print("logged in to Hugging Face")
except Exception as e:
    print(f"skipping HF login ({e}) -- fine if the model download below isn't gated")

## 2. Load both models once (not per-request)

This is the expensive part (checkpoint download + load into VRAM) -- it only
happens here, at startup, not on every API call.

In [ ]:
import sys, importlib
import torch
from diffusers import AutoPipelineForText2Image

t2i = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16")
t2i = t2i.to("cuda")
print("text-to-image model loaded")

# Same fix as the baseline notebook: import hy3dshape from inside its own
# folder (matching upstream's own usage) rather than from the repo root,
# which resolves to a hollow namespace package instead of the real package.
for mod_name in list(sys.modules):
    if mod_name == "hy3dshape" or mod_name.startswith("hy3dshape."):
        del sys.modules[mod_name]
importlib.invalidate_caches()
os.chdir("/kaggle/working/Hunyuan3D-2.1/hy3dshape")
sys.path.insert(0, os.getcwd())

import hy3dshape
assert hy3dshape.__file__ is not None, "still resolving to a namespace package -- see hunyuan3d_baseline.ipynb's troubleshooting"

from hy3dshape.rembg import BackgroundRemover
from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline

shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("tencent/Hunyuan3D-2.1")
rembg = BackgroundRemover()
print("Hunyuan3D shape model loaded")

### 2b. Load Shap-E (direct text-to-3D — the default, fast path)

Genuinely text-conditioned (`openai/shap-e` via `diffusers`) — no image step
in between. This is what a `mode: "direct"` request uses. Small and fast
(the model's own paper describes generating "complex and diverse 3D assets
in a matter of seconds"), noticeably lower detail than the image-based
Hunyuan3D path above, which stays available as `mode: "image"` for when
that's toggled on.

Unlike Hunyuan3D, this needs no custom repo clone or compiled extensions —
`diffusers`/`transformers`/`trimesh` are already installed from step 1.

In [ ]:
from diffusers import ShapEPipeline

# No variant="fp16" -- despite diffusers' own docs example showing it, the
# openai/shap-e repo doesn't actually publish fp16-variant weight files
# (fails with "no such modeling files are available"). torch_dtype alone
# still gets the fp16 memory/speed benefit, just via casting after load
# instead of downloading pre-quantized files.
shape_e_pipe = ShapEPipeline.from_pretrained("openai/shap-e", torch_dtype=torch.float16)
shape_e_pipe = shape_e_pipe.to("cuda")
print("Shap-E (direct text-to-3D) model loaded")

## 3. The two generation functions

`generate_direct()` (Shap-E, default) and `generate_image_based()`
(text -> image -> Hunyuan3D, toggle-on) both end up producing an `.stl` +
a metadata dict, so the API layer below can treat them identically. If you
hit a CUDA out-of-memory error with all three models resident, the fix is
the baseline notebook's pattern -- `del`/reload models around each use
instead of keeping everything loaded -- at the cost of extra latency per
request.

In [ ]:
import time, uuid, threading
import numpy as np
import trimesh
from pathlib import Path
from diffusers.utils import export_to_ply

OUTPUT_DIR = Path("/kaggle/working/api_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# image-based (Hunyuan3D) path only. 128 was tried first for speed and
# produced a degenerate, near-solid blob -- too coarse to resolve small/thin
# features like bolt holes and a thin plate cross-section. 256 confirmed
# real structure (tens of thousands of faces) on the same input; see
# hunyuan3d_baseline.ipynb's troubleshooting for the measurement. The
# diffusion-sampling step (50 steps, ~2 min) is a separate, fixed cost
# independent of this value -- don't expect this alone to be fast.
SHAPE_OCTREE_RESOLUTION = 256

generation_lock = threading.Lock()  # serialize GPU access across concurrent requests, regardless of mode


def generate_direct(prompt: str):
    """Text -> 3D directly via Shap-E. Default path: fast (seconds), lower detail."""
    with generation_lock:
        t0 = time.perf_counter()

        result = shape_e_pipe(
            prompt, guidance_scale=15.0, num_inference_steps=64, frame_size=256, output_type="mesh"
        ).images
        mesh_output = result[0]

        ply_path = OUTPUT_DIR / f"{uuid.uuid4().hex}.ply"
        export_to_ply(mesh_output, str(ply_path))

        mesh = trimesh.load(str(ply_path))
        # Shap-E's mesh output faces the bottom by default -- rotate upright
        # (this is the fix documented in diffusers' own Shap-E guide).
        rot = trimesh.transformations.rotation_matrix(-np.pi / 2, [1, 0, 0])
        mesh = mesh.apply_transform(rot)

        mesh_path = ply_path.with_suffix(".stl")
        mesh.export(mesh_path)
        ply_path.unlink()

        elapsed = time.perf_counter() - t0
        return mesh_path, {
            "faces": len(mesh.faces),
            "watertight": bool(mesh.is_watertight),
            "generation_time_seconds": elapsed,
        }


def generate_image_based(prompt: str):
    """Text -> image (sd-turbo) -> 3D via Hunyuan3D-2.1. Toggle-on path: a few
    minutes (see hunyuan3d_baseline.ipynb's measurements), more detail."""
    with generation_lock:
        t0 = time.perf_counter()

        image = t2i(prompt=prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
        image = rembg(image.convert("RGB"))

        mesh = shape_pipeline(image=image, octree_resolution=SHAPE_OCTREE_RESOLUTION, num_chunks=20000)[0]

        mesh_id = f"{uuid.uuid4().hex}.stl"
        mesh_path = OUTPUT_DIR / mesh_id
        mesh.export(mesh_path)  # .stl specifically -- matches the frontend's STLLoader

        elapsed = time.perf_counter() - t0
        return mesh_path, {
            "faces": len(mesh.faces),
            "watertight": bool(mesh.is_watertight),
            "generation_time_seconds": elapsed,
        }

# Smoke test -- Shap-E only (seconds, not minutes) as a fast first check.
# The image-based (Hunyuan3D) path was already validated separately -- see
# hunyuan3d_baseline.ipynb -- so it's not re-tested here to save time.
_test_path, _test_info = generate_direct("a small mechanical bracket")
print(_test_info)

## 4. API server (FastAPI) — async job queue, not a blocking request

**Important:** `/generate` here returns almost instantly with a `job_id` --
it does NOT wait for the mesh. The actual generation runs in a background
thread, and the client polls `/jobs/{job_id}` until it's done.

This is deliberate, not extra complexity for its own sake: a single HTTP
request that blocks for the entire generation time is fragile through a
public tunnel -- Cloudflare's free quick tunnels enforce a request timeout,
and a real Hunyuan3D generation (especially a cold first request with
CUDA/kernel warm-up) can run past it. When that happens the server finishes
the work successfully but the client never gets the response back ("Failed
to fetch" with no server-side error -- confusing to debug from the frontend
side alone). Polling a short-lived status endpoint sidesteps the whole
problem regardless of how long generation actually takes.

In [ ]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
import uvicorn

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

jobs = {}  # job_id -> {"status": "pending"} | {"status": "done", "result": {...}} | {"status": "error", "error": str}
jobs_lock = threading.Lock()


def _run_job(job_id: str, prompt: str, mode: str):
    try:
        if mode == "image":
            mesh_path, meta = generate_image_based(prompt)
            part_type = "hunyuan3d"
            desc = f"Hunyuan3D-2.1 (image-based) generation: \"{prompt}\" ({meta['faces']} faces, watertight={meta['watertight']})"
        else:
            mesh_path, meta = generate_direct(prompt)
            part_type = "shap-e"
            desc = f"Shap-E (direct text-to-3D) generation: \"{prompt}\" ({meta['faces']} faces, watertight={meta['watertight']})"

        result = {
            "prompt": prompt,
            "part_type": part_type,
            "part_type_inferred": False,
            "resolved_description": desc,
            "parsed_fields": [],
            "defaulted_fields": [],
            "generation_time_seconds": meta["generation_time_seconds"],
            "mesh_url": f"/mesh/{mesh_path.name}",
        }
        with jobs_lock:
            jobs[job_id] = {"status": "done", "result": result}
    except Exception as exc:
        with jobs_lock:
            jobs[job_id] = {"status": "error", "error": f"{type(exc).__name__}: {exc}"}


@app.post("/generate")
async def api_generate(request: Request):
    body = await request.json()
    prompt = (body.get("prompt") or "").strip()
    mode = (body.get("mode") or "direct").strip().lower()
    if not prompt:
        raise HTTPException(400, "prompt is empty")
    if mode not in ("direct", "image"):
        raise HTTPException(400, f"unknown mode {mode!r}, expected 'direct' or 'image'")

    job_id = uuid.uuid4().hex
    with jobs_lock:
        jobs[job_id] = {"status": "pending"}
    threading.Thread(target=_run_job, args=(job_id, prompt, mode), daemon=True).start()
    return {"job_id": job_id}


@app.get("/jobs/{job_id}")
def api_job(job_id: str):
    with jobs_lock:
        job = jobs.get(job_id)
    if job is None:
        raise HTTPException(404, "unknown job id")
    return job


@app.get("/mesh/{mesh_id}")
def api_mesh(mesh_id: str):
    path = OUTPUT_DIR / mesh_id
    if not path.is_file() or path.suffix != ".stl":
        raise HTTPException(404, "not found")
    return FileResponse(path, media_type="model/stl", filename=mesh_id)


@app.get("/health")
def health():
    return {"status": "ok"}


def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=_run, daemon=True)
server_thread.start()
time.sleep(2)
print("API server running on http://localhost:8000 (inside this Kaggle kernel -- not reachable yet, see next cell)")

## 5. Expose it publicly via a Cloudflare quick tunnel

No signup/account needed -- `cloudflared`'s quick tunnels are free and
anonymous, they just print a random `https://xxxxx.trycloudflare.com` URL
that proxies to `localhost:8000` for as long as this cell keeps running.

**Leave this cell running** (don't interrupt it) for as long as you want
your local frontend to be able to reach this backend.

In [ ]:
import subprocess, re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

public_url = None
for line in tunnel_proc.stdout:
    print(line, end="")
    m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if m and public_url is None:
        public_url = m.group(0)
        print()
        print("=" * 70)
        print(f"PUBLIC BACKEND URL: {public_url}")
        print("Paste this into the 'Backend URL' field at the top of your local")
        print("web UI (ml/cad/server.py -> http://127.0.0.1:5050) and click Generate.")
        print("=" * 70)
        print()
    # keep streaming cloudflared's log output below so you can see it's alive

## Notes

- `mode: "direct"` (Shap-E, default) — a few seconds per request, no caching.
  `mode: "image"` (Hunyuan3D, toggle-on) — a few minutes: the diffusion-
  sampling step alone (50 steps) runs ~2 minutes independent of resolution,
  plus volume decoding on top (tens of seconds at `octree_resolution=256`).
  This is why the frontend polls for the result instead of waiting on one
  request, regardless of mode.
- To stop serving: interrupt/stop the tunnel cell (step 5) and/or the kernel.
  The public URL immediately stops working.
- `SHAPE_OCTREE_RESOLUTION=256` (step 3, image-based path only) is the
  current default after `128` was confirmed too coarse (produced a
  degenerate blob instead of real geometry). Raising it further trades more
  time for more detail; lowering it back toward 128 risks losing small
  features again -- test on a mesh with a clear expected shape (like a
  bracket with holes) before trusting a lower value.
- Shap-E's direct output is noticeably lower-detail than Hunyuan3D's --
  that's the expected tradeoff for skipping the image step, not a bug.